# Notebook 02 (Participant): Evaluate Your Generated Designs

In Notebook 01 you trained a generative model and produced candidate beam designs.
Now comes the critical question: **are those designs actually any good?**

In generative modeling for engineering, "good" is **not** a single number.
A design can look plausible yet fail simulation.  It can perform well on one
objective yet violate a critical constraint.  It can be high-quality but
identical to a training example -- memorised, not generalised.

This notebook walks you through a **structured evaluation pipeline** that
diagnoses generative model quality from multiple complementary angles, each
revealing something the others miss.

## A taxonomy of generative-model metrics

```
                       How do we evaluate a generative model?
                                      |
          ┌───────────────┬───────────┼───────────────┬──────────────────┐
          |               |           |               |                  |
    Visual         Simulation    Constraint      Distributional    Diversity &
   Inspection     Performance   Satisfaction      Similarity        Coverage
          |               |           |               |                  |
   "Does it          "Does it     "Is it          "Does it          "Did we
    look right?"      work?"       legal?"         match            explore?"
                                                   reality?"
          |               |           |               |                  |
   Residual        Compliance   Volfrac error      MMD            Pairwise L2
   heatmaps        histogram    distribution     (Gaussian       DPP diversity
                   + scatter    + feasibility     kernel)        NN novelty
                   + per-sample   rate bars                      PCA embedding
                     gap bars
```

No single metric tells the whole story.  A model can ace one category and
fail another -- and *which failure matters most* depends on your application.

| Category | Question | Beams2D metric | Why it matters |
|----------|----------|---------------|----------------|
| **Visual inspection** | Does it look like a real beam? | Residual heatmaps | Quick sanity check; catches gross failures |
| **Simulation performance** | Does the physics solver confirm it works? | Compliance gap vs baseline | The ground truth -- simulation is our oracle |
| **Constraint satisfaction** | Does it obey the engineering spec? | Volume fraction error | A stiff beam using too much material is invalid |
| **Distributional similarity** | Does the generator match the real data distribution? | MMD (Maximum Mean Discrepancy) | Detects mode collapse, unrealistic densities |
| **Diversity & coverage** | Did the model explore, or did it memorise? | Pairwise L2, DPP, NN novelty | A model outputting one beam 24 times is useless |
| **Optimization warmstarting** | Does it give the optimizer a head start? | IOG, COG, FOG | The ultimate downstream utility test |

### The evaluation pipeline at a glance

```
Generated Designs    Baseline Designs    Training Designs
       |                    |                    |
       v                    v                    v
 [ Visual Inspection ]                   [ Reference set ]
       |                    |                    |
       v                    v                    |
   [ Simulate ]        [ Simulate ]              |
       |                    |                    |
       v                    v                    |
   Objectives           Objectives               |
       \                  /                      |
        \                /                       |
         v              v                        |
    Simulation Metrics                           |
    (gap, improvement rate)                      |
              |                                  |
              v                                  v
    Constraint Metrics              Distributional Metrics
    (volfrac error, feasibility)    (MMD, pixel distributions)
              |                                  |
              v                                  v
    Diversity Metrics  <────────────────  PCA Embedding
    (pairwise L2, DPP, NN novelty)
              |
              v
    Optimization Warmstarting
    (IOG, COG, FOG trajectories)
              |
              v
     Summary Dashboard
```

**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.

### Notebook map

| Part | What you do | Key output |
|------|-------------|------------|
| Setup | Install deps, load artifacts | `gen_designs`, `baseline_designs`, `conditions` |
| Part 1 | Visual inspection (the eye test) | Residual heatmaps |
| Part 2 (Fill-in 02-A) | Per-sample simulation | `results` DataFrame |
| Part 3 | Constraint satisfaction analysis | Volfrac scatter + error distribution |
| Part 4 (Fill-in 02-B) | Distributional similarity (MMD) | `mmd_value` |
| Part 5 | Diversity & coverage | Pairwise heatmap + PCA embedding |
| Part 6 | Optimization warmstarting (demo) | Trajectory plots with IOG/COG/FOG |
| Part 7 (Fill-in 02-C) | Comprehensive summary dashboard | `summary_df` |

### Legend
- `PUBLIC FILL-IN CELL` -- you write code here.
- `CHECKPOINT` -- run this assertion block to verify before moving on.
- `# START FILL` / `# END FILL` -- your edits go between these markers.

---
## Step 0: Install dependencies

In [ ]:
# Colab/local dependency bootstrap
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab

if IN_COLAB or FORCE_INSTALL:
    def pip_install(pkgs):
        subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])

    pip_install(["engibench[all]", "sqlitedict", "matplotlib", "tqdm", "tyro", "wandb"])
    pip_install(["git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"])
    try:
        import torch
    except Exception:
        pip_install(["torch", "torchvision"])
    print("Install complete.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install here.")

---
## Step 1: Load artifacts from Notebook 01

We need three files that Notebook 01 exported:
- `generated_designs.npy` -- the designs your model produced
- `baseline_designs.npy` -- optimised reference designs from the dataset
- `conditions.json` -- the boundary-condition configs for each sample

The next cell contains a recovery function that **automatically rebuilds** these
artifacts if they are missing (e.g., if you jumped straight to NB02).  You do not
need to read or understand that function -- just run it.

In [ ]:
# ── Artifact recovery (runs only if NB01 artifacts are missing) ──────────
# This cell auto-builds NB01 artifacts so NB02 works standalone.
# You do NOT need to read this code -- just run the cell.

import importlib
import json, random, sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Workshop helpers
if "google.colab" in sys.modules:
    import subprocess as _sp
    _utils = "/content/workshop_utils"
    os.makedirs(_utils, exist_ok=True)
    _branch = "codex/dcc26-workshop-notebooks"
    _base = f"https://raw.githubusercontent.com/IDEALLab/EngiOpt/{_branch}/workshops/dcc26/utils"
    for _f in ("notebook_helpers.py", "__init__.py"):
        if not os.path.exists(f"{_utils}/{_f}"):
            _sp.check_call(["wget", "-q", f"{_base}/{_f}", "-O", f"{_utils}/{_f}"])
else:
    _utils = os.path.abspath("../utils") if os.path.isdir("../utils") else "workshops/dcc26/utils"
sys.path.insert(0, _utils)
import notebook_helpers  # noqa: E402
importlib.reload(notebook_helpers)  # always pick up latest edits
from notebook_helpers import *  # noqa: F401,F403

from engibench.utils.all_problems import BUILTIN_PROBLEMS

PROBLEM_ID = "beams2d"

try:
    from engiopt.cgan_2d.cgan_2d import Generator as EngiOptCGAN2DGenerator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Could not import engiopt. Run the install cell first; on Colab, restart runtime after install."
    ) from exc


def _resolve_artifact_dir(create=False):
    p = Path("/content/dcc26_artifacts") if "google.colab" in sys.modules else Path("workshops/dcc26/artifacts")
    if create:
        p.mkdir(parents=True, exist_ok=True)
    return p


def _build_artifacts_locally(artifact_dir, seed=7, n_train=512, n_samples=24, epochs=8, batch_size=64, latent_dim=32):
    """Replicate the NB01 train+generate pipeline to produce evaluation artifacts."""
    print("Auto-building NB01 artifacts (this takes ~1 min)...")
    random.seed(seed); np.random.seed(seed); th.manual_seed(seed)
    if th.cuda.is_available(): th.cuda.manual_seed_all(seed)
    device = th.device("cuda" if th.cuda.is_available() else "cpu")
    problem = BUILTIN_PROBLEMS[PROBLEM_ID](seed=seed)
    train_ds, test_ds = problem.dataset["train"], problem.dataset["test"]
    ckeys = problem.conditions_keys
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(train_ds), size=min(n_train, len(train_ds)), replace=False)
    conds = np.stack([np.array(train_ds[k])[idx].astype(np.float32) for k in ckeys], axis=1)
    designs = np.array(train_ds["optimal_design"])[idx].astype(np.float32)
    targets = designs * 2.0 - 1.0
    model = EngiOptCGAN2DGenerator(latent_dim=latent_dim, n_conds=conds.shape[1], design_shape=problem.design_space.shape).to(device)
    opt = th.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    dl = DataLoader(TensorDataset(th.tensor(conds), th.tensor(targets)), batch_size=batch_size, shuffle=True)
    losses = []
    for ep in range(epochs):
        model.train(); ep_loss = 0.0
        for cb, tb in dl:
            cb, tb = cb.to(device), tb.to(device)
            pred = model(th.randn(cb.shape[0], latent_dim, device=device), cb)
            loss = crit(pred, tb); opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item()
        avg = ep_loss / len(dl); losses.append(avg)
        print(f"  epoch {ep+1:02d}/{epochs} loss={avg:.4f}")
    sc = min(n_samples, len(test_ds))
    sel = rng.choice(len(test_ds), size=sc, replace=False)
    tc = np.stack([np.array(test_ds[k])[sel].astype(np.float32) for k in ckeys], axis=1)
    bl = np.array(test_ds["optimal_design"])[sel].astype(np.float32)
    model.eval()
    with th.no_grad():
        out = model(th.randn(sc, latent_dim, device=device), th.tensor(tc, device=device))
        gd = ((out.clamp(-1, 1) + 1) / 2).clamp(0, 1).cpu().numpy().astype(np.float32)
    cond_recs = []
    for i in range(sc):
        rec = {}
        for j, k in enumerate(ckeys):
            rec[k] = bool(tc[i, j]) if k == "overhang_constraint" else float(tc[i, j])
        cond_recs.append(rec)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    np.save(artifact_dir / "generated_designs.npy", gd)
    np.save(artifact_dir / "baseline_designs.npy", bl)
    with open(artifact_dir / "conditions.json", "w") as f: json.dump(cond_recs, f, indent=2)
    pd.DataFrame({"epoch": range(1, len(losses)+1), "train_loss": losses}).to_csv(artifact_dir / "training_history.csv", index=False)
    th.save({"model": model.state_dict(), "condition_keys": ckeys, "latent_dim": latent_dim}, artifact_dir / "engiopt_cgan2d_generator_supervised.pt")
    print("Artifacts ready at", artifact_dir)


ARTIFACT_DIR = _resolve_artifact_dir(create=True)
_required = [ARTIFACT_DIR / f for f in ("generated_designs.npy", "baseline_designs.npy", "conditions.json")]
if not all(p.exists() for p in _required):
    _build_artifacts_locally(ARTIFACT_DIR)

print("Artifact directory:", ARTIFACT_DIR)

In [ ]:
# ── Load artifacts ───────────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

gen_designs = np.load(ARTIFACT_DIR / "generated_designs.npy")
baseline_designs = np.load(ARTIFACT_DIR / "baseline_designs.npy")
with open(ARTIFACT_DIR / "conditions.json") as f:
    conditions = json.load(f)

print(f"Generated designs : {gen_designs.shape}  (values in [{gen_designs.min():.2f}, {gen_designs.max():.2f}])")
print(f"Baseline designs  : {baseline_designs.shape}")
print(f"Condition records : {len(conditions)}")
print(f"Condition keys    : {list(conditions[0].keys())}")

In [ ]:
# Load a reference subset of training designs for distributional + novelty metrics
problem_ref = BUILTIN_PROBLEMS[PROBLEM_ID](seed=7)
train_designs_full = np.array(problem_ref.dataset["train"]["optimal_design"]).astype(np.float32)
ref_idx = np.random.default_rng(7).choice(
    len(train_designs_full), size=min(1024, len(train_designs_full)), replace=False
)
train_reference = train_designs_full[ref_idx]
print(f"Training reference set: {train_reference.shape[0]} designs")

---
## Part 1: Visual Inspection -- The Eye Test

Before computing any metric, **look at the designs**.  Visual inspection catches
gross failures immediately: is the model producing solid blocks? random noise?
something that looks vaguely beam-like?

We show three views:
1. **Side-by-side gallery** -- generated vs optimised baseline
2. **Pixel residual heatmaps** -- where exactly do the designs differ?

Visual inspection is *necessary* but **not sufficient**.  A design can look
plausible yet perform terribly in simulation, or violate constraints that
are invisible to the eye.  The rest of this notebook quantifies what your
eyes cannot.

In [ ]:
show_residual_heatmaps(gen_designs, baseline_designs, n_show=6)

**Takeaway:** The residual heatmaps reveal where the generator struggles most.
Bright regions = large pixel error.  Notice how errors tend to cluster at
structural boundaries and fine features -- exactly the details that matter
most for physical performance.

But pixels alone don't tell us about *compliance*, *constraint violations*, or
*diversity*.  We need simulation.

---
## Part 2: Simulation Performance -- "Does it work?"

The **physics simulator** is our oracle.  For Beams2D, it computes the
*compliance* of each design under the given boundary conditions:
- **Lower compliance = stiffer beam = better design**

We simulate both the generated design and its corresponding baseline
(the optimised design from the dataset) under **identical conditions**.
The difference tells us how far the generator is from optimal.

> **Analogy:** Imagine you asked an architecture student to sketch a bridge.
> Visual inspection tells you the sketch looks bridge-like.  But only a
> structural engineer (our simulator) can tell you whether it would actually
> stand up.

In [ ]:
problem = BUILTIN_PROBLEMS[PROBLEM_ID](seed=7)

# Feasibility tolerance: how close must volfrac be to the target?
VOLFRAC_TOL = 0.05

# PUBLIC FILL-IN CELL 02-A
# Goal: build a list of dicts, one per sample, with objective + feasibility info.
#
# For each sample i, you have:
#   g = gen_designs[i]          -- generated design (2D numpy array)
#   b = baseline_designs[i]     -- baseline design (2D numpy array)
#   cfg = conditions[i]         -- dict with keys like 'volfrac', 'rmin', etc.

rows = []

# START FILL ---------------------------------------------------------------
for i in range(len(gen_designs)):
    g = gen_designs[i]
    b = baseline_designs[i]
    cfg = dict(conditions[i])

    # 1) Compute volume fractions (mean pixel value of each design)
    g_vf = None  # TODO: compute mean of g
    b_vf = None  # TODO: compute mean of b
    target_vf = cfg["volfrac"]

    # 2) Check feasibility: is |actual_vf - target_vf| <= VOLFRAC_TOL?
    g_feasible = None  # TODO: True/False
    b_feasible = None  # TODO: True/False

    # 3) Simulate both designs under identical conditions
    #    Hint: call problem.reset(seed=...) before each simulate for reproducibility
    #    Hint: problem.simulate(design, config=cfg) returns an array; take element [0]
    problem.reset(seed=7 + i)
    g_obj = None  # TODO: simulate the generated design
    problem.reset(seed=7 + i)
    b_obj = None  # TODO: simulate the baseline design

    # 4) Record everything
    rows.append({
        "sample": i,
        "gen_obj": g_obj,
        "base_obj": b_obj,
        "gen_minus_base": g_obj - b_obj,
        "gen_volfrac": g_vf,
        "target_volfrac": target_vf,
        "gen_feasible": g_feasible,
        "base_feasible": b_feasible,
    })

raise NotImplementedError("Fill in the TODOs above, then delete this line.")
# END FILL -----------------------------------------------------------------

results = pd.DataFrame(rows)
results.head()

In [ ]:
# CHECKPOINT 02-A
expected_cols = {"sample", "gen_obj", "base_obj", "gen_minus_base", "gen_volfrac",
                 "target_volfrac", "gen_feasible", "base_feasible"}
missing_cols = expected_cols - set(results.columns)
assert not missing_cols, f"Missing columns: {missing_cols}"
assert len(results) == len(gen_designs), f"Expected {len(gen_designs)} rows, got {len(results)}"
assert results["gen_obj"].notna().all(), "gen_obj contains NaN -- did you forget to simulate?"
assert results["gen_feasible"].dtype == bool, "gen_feasible should be boolean"
print(f"Checkpoint 02-A passed: {len(results)} samples evaluated.")
print(f"  Feasible generated: {results['gen_feasible'].sum()}/{len(results)}")
print(f"  Feasible baseline:  {results['base_feasible'].sum()}/{len(results)}")

### Visualising simulation results

Three complementary views:
1. **Histogram** -- overall distribution of objectives (generated vs baseline)
2. **Scatter plot** -- per-sample pairing (points below diagonal = generated is better)
3. **Residual bar chart** -- per-sample gap, signed (green = generated outperforms)

In [ ]:
show_objective_comparison(results)

In [ ]:
show_objective_residuals(results)

### Reading the simulation results

- **Histogram overlap**: If the blue (generated) and orange (baseline) distributions
  overlap heavily, the generator is competitive.  If blue is shifted right (higher
  compliance), the generator produces weaker designs.

- **Scatter diagonal**: Points *below* the diagonal line mean the generated design
  outperformed the optimised baseline for that sample -- a strong result.

- **Residual bars**: The bar chart makes the per-sample gap immediately visible.
  Consistent green bars = the model is competitive.  Large red bars = specific
  failure modes worth investigating (check the design images for those samples).

---
## Part 3: Constraint Satisfaction -- "Is it legal?"

A design that performs well but **violates constraints** is useless in practice.
For Beams2D, the key constraint is **volume fraction**: the design must use
a specific amount of material (neither too much nor too little).

> **Analogy:** An architect who designs a beautiful building that exceeds the
> budget by 50% has not solved the problem -- they have created a new one.

We already computed `gen_volfrac` and `target_volfrac` in the simulation loop.
Now let's visualise how well the generator satisfies this constraint.

In [ ]:
show_volfrac_analysis(results, volfrac_tol=VOLFRAC_TOL)

In [ ]:
show_feasibility_bars(results)

### Reading the constraint results

- **Left scatter**: Points near the diagonal are feasible; points far from it
  are violating the volume fraction constraint.  The green band shows the
  tolerance window.

- **Error histogram**: A narrow distribution centered at zero means the generator
  has learned to control material usage.  A wide or biased distribution suggests
  the model ignores the volume fraction condition.

- **Feasibility rate**: The bar chart gives the bottom line.  If the baseline
  achieves ~100% feasibility but the generator is at 50%, there is a clear
  conditioning failure.

**Why this matters beyond beams:**  In real engineering, constraints can be
stress limits, manufacturing tolerances, thermal budgets, or regulatory
requirements.  A generative model that ignores constraints generates
*interesting but unusable* designs.

---
## Part 4: Distributional Similarity -- "Does it match reality?"

The previous metrics evaluated designs **individually** (per-sample objective,
per-sample feasibility).  But we also need to ask: does the *distribution*
of generated designs match the distribution of the ground-truth optimal designs
**for the same conditions**?

### What is MMD?

**Maximum Mean Discrepancy (MMD)** is a kernel-based distance between two
distributions.  Intuitively:

1. Map each design into a high-dimensional feature space via a Gaussian kernel
2. Compare the *mean embeddings* of the two sets
3. If the means match, the distributions are similar; if they diverge, they are different

$$\text{MMD}^2 = \underbrace{\mathbb{E}[k(x, x')]}_{\text{gen-gen similarity}} + \underbrace{\mathbb{E}[k(y, y')]}_{\text{base-base similarity}} - 2\,\underbrace{\mathbb{E}[k(x, y)]}_{\text{cross similarity}}$$

- **MMD = 0**: generated and baseline distributions are identical
- **MMD > 0**: they differ (larger = more different)
- The kernel bandwidth $\sigma$ controls the scale of comparison

### Why compare generated vs baseline?

Our generator is **conditional** -- it takes test conditions and produces
designs.  The baseline contains the ground-truth optima for those *same*
test conditions.  Comparing generated vs baseline directly measures whether
the generator has learned to produce the right designs for the right conditions.

### Choosing sigma without test-data leakage

The Gaussian kernel bandwidth $\sigma$ determines what scale of difference
the kernel is sensitive to.  We set it using the **median heuristic** on the
*training data only* -- the median pairwise distance among training designs.
This avoids leaking test information into the metric while ensuring the
kernel operates in a meaningful range.

### Why MMD and not just "average quality"?

A model could produce 24 copies of the single best design.  Per-sample metrics
would look great!  But the *distribution* would be nothing like the diverse
baseline set.  MMD catches this.

In [ ]:
# Visual intuition: where does each set place material?
show_spatial_distribution_comparison(gen_designs, baseline_designs, train_reference)


In [ ]:
# PUBLIC FILL-IN CELL 02-B
# Goal: compute MMD between generated designs and baseline designs (same conditions).
#
# MMD uses a Gaussian (RBF) kernel: k(x,y) = exp(-||x-y||^2 / (2*sigma^2))
#
# We set sigma from the TRAINING data (median heuristic) to avoid test-data leakage,
# then apply that fixed sigma to the gen-vs-baseline comparison.
#
# You have:
#   gen_designs      -- (N, H, W) numpy array of generated designs
#   baseline_designs -- (N, H, W) numpy array of optimized designs (same conditions)
#   train_reference  -- (M, H, W) numpy array of training designs
#   cdist            -- from scipy.spatial.distance (already imported)
#
# Steps:
#   1. Flatten all design sets to 2D
#   2. Compute sigma from training pairwise distances (median heuristic)
#   3. Compute pairwise squared distances between generated and baseline
#   4. Apply Gaussian kernel: K = exp(-D / (2 * sigma^2))
#   5. MMD = mean(K_gg) + mean(K_bb) - 2 * mean(K_gb)

# START FILL ---------------------------------------------------------------
# 1. Flatten
gen_flat = None   # TODO: reshape gen_designs to (N, H*W)
base_flat = None  # TODO: reshape baseline_designs to (N, H*W)
ref_flat = None   # TODO: reshape train_reference to (M, H*W)

# 2. Sigma from training data only (no test leakage)
#    Hint: compute pairwise sqeuclidean distances within ref_flat,
#    then sigma = sqrt(median of those distances)
D_ref = None      # TODO: cdist(ref_flat, ref_flat, "sqeuclidean")
sigma = None      # TODO: float(np.sqrt(np.median(D_ref)))

# 3. Pairwise squared distances for gen vs baseline
D_gg = None  # TODO: cdist(gen_flat, gen_flat, "sqeuclidean")
D_bb = None  # TODO: cdist(base_flat, base_flat, "sqeuclidean")
D_gb = None  # TODO: cdist(gen_flat, base_flat, "sqeuclidean")

# 4. Gaussian kernel
K_gg = None  # TODO: np.exp(-D_gg / (2 * sigma**2))
K_bb = None  # TODO: same for D_bb
K_gb = None  # TODO: same for D_gb

# 5. MMD
mmd_value = None  # TODO: float(K_gg.mean() + K_bb.mean() - 2 * K_gb.mean())

raise NotImplementedError("Fill in the TODOs above, then delete this line.")
# END FILL -----------------------------------------------------------------

print(f"Sigma (median heuristic on training data): {sigma:.2f}")
print(f"MMD(generated, baseline) = {mmd_value:.6f}")
print(f"  K_gg mean (gen-gen similarity):    {K_gg.mean():.6f}")
print(f"  K_bb mean (base-base similarity):  {K_bb.mean():.6f}")
print(f"  K_gb mean (cross similarity):      {K_gb.mean():.6f}")

In [ ]:
# CHECKPOINT 02-B
assert mmd_value is not None, "mmd_value is None -- did you compute it?"
assert isinstance(mmd_value, float), "mmd_value should be a float"
assert mmd_value >= 0, f"MMD should be non-negative, got {mmd_value}"
assert sigma is not None and sigma > 1, f"sigma should be > 1 for 10k-dim data (got {sigma}); did you use the median heuristic?"
assert K_gg is not None and K_gb is not None, "Kernel matrices not computed"
print(f"Checkpoint 02-B passed: MMD = {mmd_value:.6f} (sigma = {sigma:.2f})")

In [ ]:
# Put MMD in context: compare against reference points (same training-derived sigma)
#
# 1. Training sample vs baseline: what you'd get by grabbing random training
#    designs instead of conditioning on the test conditions.
# 2. Random noise vs baseline: worst case (meaningless generator).

rng_mmd = np.random.default_rng(42)

# Training sample vs baseline (no conditioning)
train_sample_idx = rng_mmd.choice(len(train_reference), size=len(baseline_designs), replace=False)
train_sample_flat = train_reference[train_sample_idx].reshape(len(baseline_designs), -1)
D_tt = cdist(train_sample_flat, train_sample_flat, "sqeuclidean")
D_tb = cdist(train_sample_flat, base_flat, "sqeuclidean")
mmd_train_base = float(
    np.exp(-D_tt / (2*sigma**2)).mean()
    + K_bb.mean()
    - 2 * np.exp(-D_tb / (2*sigma**2)).mean()
)

# Random noise vs baseline
random_designs = rng_mmd.random(gen_designs.shape).astype(np.float32)
rand_flat = random_designs.reshape(random_designs.shape[0], -1)
D_rr = cdist(rand_flat, rand_flat, "sqeuclidean")
D_rb = cdist(rand_flat, base_flat, "sqeuclidean")
mmd_random_base = float(
    np.exp(-D_rr / (2*sigma**2)).mean()
    + K_bb.mean()
    - 2 * np.exp(-D_rb / (2*sigma**2)).mean()
)

print(f"MMD reference points (sigma={sigma:.2f}, from training data):")
print(f"  Generated vs Baseline:      {mmd_value:.6f}  (our model)")
print(f"  Train sample vs Baseline:   {mmd_train_base:.6f}  (no conditioning)")
print(f"  Random noise vs Baseline:   {mmd_random_base:.6f}  (worst case)")

show_mmd_comparison_bar(mmd_value, mmd_train_base, mmd_random_base)

In [ ]:
# PCA embedding: where do generated designs live relative to training data?
show_embedding_scatter(gen_designs, baseline_designs, train_reference)

### Reading the distributional similarity results

- **Mean design images**: If the generated mean image looks similar to the
  baseline mean image, the model has learned where material should go on
  average for these conditions.  Differences reveal spatial biases.

- **Volume fraction distributions**: If the generated distribution is narrower
  or shifted relative to baseline, the model isn't capturing the full range
  of volume fractions needed for these test conditions.

- **MMD in context**: The comparison bar chart places the generator's MMD on
  a meaningful scale:
  - **Train sample vs Baseline** (retrieval baseline): What you'd get by
    grabbing random training designs instead of conditioning.  If the
    generator beats this, it has genuinely learned to condition.
  - **Random vs Baseline** (worst case): Uniform noise -- the floor for
    a non-functional generator.
  A generator close to zero has matched the baseline distribution.  A
  generator near the train-sample bar is no better than memorising
  training data without using the conditions.

- **PCA embedding**: If generated designs (blue) cluster tightly in one
  corner while training data (grey) spans a wide region, the model has
  **mode collapse**.  Ideally, blue points should overlap with the orange
  baseline points (same conditions) while spanning a similar spread.

---
## Part 5: Diversity & Coverage -- "Did we explore?"

A generative model should produce **varied** designs, not 24 copies of the same
beam.  We measure two complementary aspects:

### Diversity (intra-set variation)
How different are the generated designs from *each other*?
- **Pairwise L2 distance**: Average Euclidean distance between all pairs of
  generated designs.  Higher = more diverse.
- **DPP diversity**: Determinantal Point Process log-determinant of the
  similarity matrix.  Captures both volume and spread of the set.

### Novelty (distance to training data)
How different are the generated designs from the *training set*?
- **Nearest-neighbour distance**: For each generated design, find the closest
  training example.  If NN distance is near zero, the model may be memorising.
  Higher = more novel.

> **The diversity-quality trade-off:**  A model that generates random noise
> would score very high on diversity but terribly on quality.  We want designs
> that are diverse *and* feasible *and* performant.  This is the fundamental
> tension in generative model evaluation.

In [ ]:
show_pairwise_distance_heatmap(gen_designs)

In [ ]:
# Compute diversity and novelty metrics
diversity_l2 = mean_pairwise_l2(gen_designs)
novelty_nn = mean_nn_distance_to_reference(gen_designs, train_reference)

# Also compute for baseline as a reference point
baseline_diversity = mean_pairwise_l2(baseline_designs)
baseline_novelty = mean_nn_distance_to_reference(baseline_designs, train_reference)

print("Diversity (mean pairwise L2):")
print(f"  Generated:  {diversity_l2:.2f}")
print(f"  Baseline:   {baseline_diversity:.2f}")
print()
print("Novelty (mean NN distance to training):")
print(f"  Generated:  {novelty_nn:.2f}")
print(f"  Baseline:   {baseline_novelty:.2f}")
print()
if diversity_l2 < baseline_diversity * 0.5:
    print("Warning: Generated diversity is much lower than baseline -- possible mode collapse.")
elif diversity_l2 > baseline_diversity * 1.5:
    print("Note: Generated diversity exceeds baseline -- check if the extra variation is meaningful.")
else:
    print("Generated diversity is comparable to baseline diversity.")

### Reading the diversity results

- **Pairwise heatmap**: Uniform warm colours = good diversity (all designs differ
  from each other).  A block of cool/dark colours = a cluster of near-identical
  designs (partial mode collapse).

- **Diversity vs baseline**: The baseline designs come from an optimiser run on
  diverse conditions, so they naturally vary.  If the generator's diversity is
  much lower, it is producing less variety than the problem demands.

- **Novelty**: Very low NN distance means the generator is reproducing training
  examples almost exactly.  Some proximity is expected (it learned from them),
  but near-zero distance suggests memorisation rather than generalisation.

---
## Part 6: Optimization Warmstarting -- "Does it speed up search?"

The **ultimate downstream test** for a generative model in engineering design:
if we use its output as a *starting point* for topology optimisation, does the
optimiser converge faster or find better solutions than starting from scratch?

### The optimality gap metrics

Starting from a generated design, we run the problem's optimiser and track the
objective at each step:

- **IOG (Initial Optimality Gap)** = objective at step 0 minus baseline optimum.
  *How good is the starting point?*

- **FOG (Final Optimality Gap)** = objective at final step minus baseline optimum.
  *How good is the final result?*

- **COG (Cumulative Optimality Gap)** = sum of all per-step gaps.
  *How much total "wasted effort" occurred across the trajectory?*
  The shaded area in the trajectory plot.

```
Objective
  ^
  |  *                         IOG = obj[0] - baseline
  |   \  *
  |    \   *  *               Shaded area = COG
  |     \       *  *  *
  |      ─ ─ ─ ─ ─ ─ ─ ─     FOG = obj[-1] - baseline
  |  - - - - - - - - - - -    ← baseline (optimised reference)
  └────────────────────────> Step
```

- IOG < 0 is ideal: the generated design is *already better* than the baseline
- FOG ≈ 0: the optimiser recovers to baseline quality regardless of start
- Small COG: the optimiser converges quickly from this warmstart

In [ ]:
# DEMO: Optimization warmstarting on a small subset (3 samples)
# This runs the EngiBench optimiser from each generated design and tracks the trajectory.
# We use only 3 samples because optimization is slower than simulation.

problem_opt = BUILTIN_PROBLEMS[PROBLEM_ID](seed=7)
n_opt_demo = min(3, len(gen_designs))
opt_data = []

for i in range(n_opt_demo):
    cfg = dict(conditions[i])

    # Run optimiser from generated design
    problem_opt.reset(seed=7 + i)
    _, opt_history = problem_opt.optimize(gen_designs[i], config=cfg)

    # Get baseline objective for reference
    problem_opt.reset(seed=7 + i)
    base_obj = float(problem_opt.simulate(baseline_designs[i], config=cfg)[0])

    # Extract objective trajectory
    obj_trajectory = [float(step.obj_values) for step in opt_history]

    opt_data.append({
        "sample_idx": i,
        "obj_trajectory": obj_trajectory,
        "base_obj": base_obj,
    })
    iog = obj_trajectory[0] - base_obj
    fog = obj_trajectory[-1] - base_obj
    cog = sum(o - base_obj for o in obj_trajectory)
    print(f"Sample {i}: IOG={iog:.1f}  FOG={fog:.1f}  COG={cog:.1f}  ({len(opt_history)} steps)")

print(f"\nOptimization complete for {n_opt_demo} samples.")

In [ ]:
show_optimization_trajectories(opt_data)

### Reading the optimization trajectories

- **Steep drop at step 0→1**: The generated design was far from optimal, but the
  optimiser quickly improved it.  This still counts as a useful warmstart if
  the total trajectory (COG) is shorter than starting from scratch.

- **Flat trajectory near baseline**: The generated design was already near-optimal
  and the optimiser had little work to do.  Best-case scenario.

- **Trajectory above baseline throughout**: The generated design was so far from
  optimal that even after optimisation it never reached baseline quality.  This
  suggests the model is producing designs in the wrong region of design space.

**In practice**, you would run this on many more samples and average the IOG/COG/FOG
to get statistically robust estimates.  For the workshop, 3 samples illustrate
the concept.

---
## Part 7: Putting It All Together

Now we aggregate all the metrics from Parts 2-6 into a single summary table.
This is the kind of table you would report in a paper or use to compare
different generative models.

In [ ]:
# PUBLIC FILL-IN CELL 02-C
# Goal: build a comprehensive summary dict and wrap it in a DataFrame.
#
# You have:
#   results          -- per-sample DataFrame from Part 2
#   mmd_value        -- MMD from Part 4
#   diversity_l2     -- from Part 5
#   novelty_nn       -- from Part 5
#   opt_data         -- optimization results from Part 6

# START FILL ---------------------------------------------------------------
# Compute average IOG/FOG/COG from the optimization demo
avg_iog = None  # TODO: mean of (first obj - base_obj) across opt_data
avg_fog = None  # TODO: mean of (last obj - base_obj) across opt_data
avg_cog = None  # TODO: mean of (sum of gaps) across opt_data

summary = {
    # Simulation performance
    "n_samples":              len(results),
    "gen_obj_mean":           None,  # TODO: mean of gen_obj column
    "base_obj_mean":          None,  # TODO: mean of base_obj column
    "objective_gap_mean":     None,  # TODO: mean of gen_minus_base column
    "improvement_rate":       None,  # TODO: fraction where gen_obj < base_obj
    # Constraint satisfaction
    "gen_feasible_rate":      None,  # TODO: fraction of feasible generated designs
    "base_feasible_rate":     None,  # TODO: fraction of feasible baseline designs
    "gen_violation_ratio":    None,  # TODO: 1 - gen_feasible_rate
    "base_violation_ratio":   None,  # TODO: 1 - base_feasible_rate
    # Distributional similarity
    "mmd":                    mmd_value,
    # Diversity & novelty
    "gen_diversity_l2":       diversity_l2,
    "gen_novelty_to_train_l2": novelty_nn,
    # Optimization warmstarting (from demo subset)
    "avg_iog":                avg_iog,
    "avg_fog":                avg_fog,
    "avg_cog":                avg_cog,
}

summary_df = None  # TODO: pd.DataFrame([summary])

raise NotImplementedError("Fill in the TODOs above, then delete this line.")
# END FILL -----------------------------------------------------------------

# Show the summary transposed for readability (one metric per row)
display(summary_df.T.rename(columns={0: "value"}))

In [ ]:
# CHECKPOINT 02-C
assert "summary_df" in dir() and summary_df is not None, "Define summary_df"
assert len(summary_df) == 1, "summary_df should have exactly one row"
required_keys = {
    "n_samples", "gen_obj_mean", "base_obj_mean", "objective_gap_mean",
    "improvement_rate", "gen_feasible_rate", "base_feasible_rate",
    "gen_violation_ratio", "base_violation_ratio",
    "mmd", "gen_diversity_l2", "gen_novelty_to_train_l2",
    "avg_iog", "avg_fog", "avg_cog",
}
missing = required_keys - set(summary_df.columns)
assert not missing, f"Missing summary columns: {missing}"
assert summary_df["gen_obj_mean"].notna().all(), "gen_obj_mean is NaN"
print("Checkpoint 02-C passed: comprehensive summary table is complete.")

In [ ]:
show_metric_summary_dashboard(summary)

---
## Export artifacts

In [ ]:
results_path = ARTIFACT_DIR / "per_sample_metrics.csv"
summary_path = ARTIFACT_DIR / "metrics_summary.csv"

results.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)

# Save objective histogram
hist_path = ARTIFACT_DIR / "objective_histogram.png"
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(results["gen_obj"], bins=10, alpha=0.7, label="Generated", color="#4C72B0")
ax.hist(results["base_obj"], bins=10, alpha=0.7, label="Baseline", color="#DD8452")
ax.set_xlabel("Compliance (lower is better)")
ax.set_ylabel("Count")
ax.set_title("Generated vs baseline objective distribution")
ax.legend()
fig.tight_layout()
fig.savefig(hist_path, dpi=150)
plt.close(fig)

# Save scatter plot
scatter_path = ARTIFACT_DIR / "objective_scatter.png"
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(results["base_obj"], results["gen_obj"], alpha=0.8)
lo = min(results["base_obj"].min(), results["gen_obj"].min()) * 0.9
hi = max(results["base_obj"].max(), results["gen_obj"].max()) * 1.1
ax.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1)
ax.set_xlabel("Baseline compliance")
ax.set_ylabel("Generated compliance")
ax.set_title("Per-sample objective comparison")
fig.tight_layout()
fig.savefig(scatter_path, dpi=150)
plt.close(fig)

# Save design grid
grid_path = ARTIFACT_DIR / "design_grid.png"
fig, axes_grid = plt.subplots(2, min(6, len(gen_designs)), figsize=(14, 5))
for i in range(min(6, len(gen_designs))):
    axes_grid[0, i].imshow(gen_designs[i], cmap="gray", vmin=0, vmax=1)
    axes_grid[0, i].set_title(f"gen {i}", fontsize=9)
    axes_grid[0, i].axis("off")
    axes_grid[1, i].imshow(baseline_designs[i], cmap="gray", vmin=0, vmax=1)
    axes_grid[1, i].set_title(f"base {i}", fontsize=9)
    axes_grid[1, i].axis("off")
fig.tight_layout()
fig.savefig(grid_path, dpi=150)
plt.close(fig)

print("Exported:")
for p in [results_path, summary_path, hist_path, scatter_path, grid_path]:
    print(f"  {p}")

---
## Discussion prompts

Use these questions to prepare for the workshop breakout discussion. There are no
"right" answers -- the goal is to develop your own informed perspective.

1. **Which metric category matters most for your domain?**  In safety-critical
   applications (aerospace, medical devices), constraint satisfaction is a hard
   requirement.  In early-stage concept exploration, diversity might matter more.
   What about your own research area?

2. **When do metrics disagree?**  A model might score well on MMD (distributional
   match) but poorly on per-sample objective (simulation performance).  What does
   that disagreement tell you?  Which metric would you trust more?

3. **Is diversity always good?**  A model that produces wildly different designs
   scores high on diversity -- but some of those designs might be nonsensical.
   When does high diversity indicate a problem rather than a strength?

4. **The warmstarting test.**  If a model's IOG is poor (bad starting points) but
   FOG is near zero (optimiser recovers), is the model useful?  What if IOG is
   great but the optimiser diverges (FOG increases)?

5. **When would you trust these results for a paper?**  We evaluated 24 samples
   with a model trained for 8 epochs on 512 examples.  What would need to change
   to make these numbers publication-ready?  (Think: sample size, training budget,
   statistical significance, multiple seeds.)

6. **Objective vs feasibility trade-off.**  If your model produces designs with
   great compliance but poor volume-fraction adherence, is that progress or a
   failure?  How would you communicate this nuance in a benchmark table?

---
## Reflection: what did you learn in NB02?

Before closing, write down your answers to these prompts:

1. **What do the metrics tell you about your model?**  Look at your summary table.
   Where does the generator excel, and where does it fall short?  Which metric
   surprised you most?

2. **Which visualisation was most informative?**  Was it the residual heatmaps,
   the PCA embedding, the optimization trajectories, or something else?  Why?

3. **What would a full benchmark study add?**  A complete EngiBench evaluation
   would test across multiple problems, multiple seeds, larger sample sizes, and
   the full metric suite (MMD, DPP, IOG/COG/FOG, violation ratio).  How would
   that change your confidence in the conclusions?

4. **How would you improve the generator?**  Based on the diagnostic pattern you
   see (which categories are strong vs weak), what would you change about the
   model architecture, training procedure, or data pipeline?

---
## Troubleshooting

If a section fails, do not continue downstream.  Fix the failing cell first, then
rerun it and its checkpoint before moving on.  The notebook is staged so that
failures are localised.